# Notebook 04 — Random Forest
## The Wisdom of Many Trees

**Dataset**: Titanic passengers
**Question we answer**: *Did this passenger survive?*
**Learning type**: Supervised Ensemble Classification — many trees voting

---

## Section 1 — What Is Random Forest?

### In plain English

One doctor can be wrong. What if you asked **100 doctors**, each trained slightly differently, and took a vote?

> Dr. Smith says survived. Dr. Jones says survived. Dr. Lee says not survived.
> 67 out of 100 say survived → **final answer: Survived**.

That is Random Forest. Each doctor is a Decision Tree.
The forest is an **ensemble** of hundreds of trees, each seeing a slightly different picture.

### Two sources of randomness (why trees differ)

1. **Bootstrap sampling** — each tree is trained on a random sample of passengers
   (drawn with replacement: some passengers appear twice, some not at all)

2. **Random feature subsets** — at each split, only a random subset of features is considered
   (forces each tree to rely on different combinations of features)

Because each tree makes different mistakes, those mistakes **cancel out** when you vote.
The consistent signal (sex, pclass, age really do predict survival) survives the vote.

## Section 2 — How Does It Learn?

### Building the forest

Repeat N times (e.g. 100):
1. Draw a bootstrap sample from the training set
2. Build a full decision tree on that sample
3. At each node, consider only `√(n_features)` randomly chosen features

### Predicting

For each new passenger, every tree votes: survived or not?
**Majority vote** → final prediction.

### Why does this beat a single tree?

| | Single Decision Tree | Random Forest |
|---|---|---|
| Overfitting | High risk | Low risk |
| Variance | High — sensitive to training data | Low — averaged out |
| Interpretability | Very high (print the tree) | Medium (feature importances only) |
| Accuracy on Titanic | ~78–80% | ~82–84% |

> A single deep tree memorises noise in the training data.
> 100 trees each memorise **different** noise → noise averages to zero.

## Section 3 — What Does the Data Need?

Random Forest is a collection of decision trees. **Same prep as a single tree.**

| Requirement | Logistic Regression | Decision Tree | **Random Forest** |
|---|---|---|---|
| Feature scaling | Required | Not needed | **Not needed** |
| Numeric encoding | Required | Required | Required |
| Missing values | Not allowed | Not allowed | Not allowed (sklearn) |
| Distribution shape | Matters | Irrelevant | **Irrelevant** |

Random Forest tolerates outliers, skewed distributions, and correlated features well.
The two things it does need: **no raw strings** and **no NaN values**.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split

sns.set_theme(style='whitegrid')
plt.rcParams['figure.dpi'] = 110

df_raw = sns.load_dataset('titanic')

df = df_raw.copy()
df['age']          = df['age'].fillna(df['age'].median())
df['fare']         = df['fare'].fillna(df['fare'].median())
df['embarked']     = df['embarked'].fillna(df['embarked'].mode()[0])
df['sex_enc']      = (df['sex'] == 'female').astype(int)
df['embarked_enc'] = df['embarked'].map({'S': 0, 'C': 1, 'Q': 2})
df['family_size']  = df['sibsp'] + df['parch'] + 1
df['log_fare']     = np.log1p(df['fare'])
df = df.dropna(subset=['survived'])

FEATURES = ['pclass','sex_enc','age','sibsp','parch','log_fare','embarked_enc','family_size']
TARGET = 'survived'

X = df[FEATURES].values
y = df[TARGET].values.astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

print(f"Train: {len(X_train)} passengers  |  Test: {len(X_test)} passengers")
print("No scaling applied — Random Forest does not need it.")

## Section 4 — Train the Model and Read the Rules

Random Forest has no single tree to read.
Instead we look at **feature importances** — how much each feature reduced impurity,
averaged across all trees.

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

single_tree = DecisionTreeClassifier(max_depth=None, random_state=42)
single_tree.fit(X_train, y_train)
acc_tree = accuracy_score(y_test, single_tree.predict(X_test))

rf = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)
acc_rf = accuracy_score(y_test, rf.predict(X_test))

print(f"Single Decision Tree (unlimited depth) : {acc_tree*100:.1f}%")
print(f"Random Forest (100 trees)              : {acc_rf*100:.1f}%")
print()
print(f"Improvement: +{(acc_rf - acc_tree)*100:.1f} percentage points")
print()
print("The forest is more reliable because individual tree errors cancel out.")

In [ ]:
importance_df = pd.DataFrame({
    'Feature'    : FEATURES,
    'Importance' : rf.feature_importances_.round(4),
}).sort_values('Importance', ascending=True)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))

ax1.barh(importance_df['Feature'], importance_df['Importance'], color='#26A69A')
ax1.set_title('Random Forest Feature Importances\n(averaged across 100 trees)', fontsize=12)
ax1.set_xlabel('Mean Gini importance', fontsize=11)

names  = ['Single\nDecision Tree', 'Random\nForest (100)']
accs   = [acc_tree*100, acc_rf*100]
colors = ['#FFA726', '#26A69A']
bars   = ax2.bar(names, accs, color=colors, width=0.4)
ax2.set_ylim(60, 95)
ax2.set_ylabel('Test accuracy (%)', fontsize=11)
ax2.set_title('Single Tree vs Random Forest', fontsize=12)
for bar, acc in zip(bars, accs):
    ax2.text(bar.get_x() + bar.get_width()/2, acc + 0.5, f'{acc:.1f}%',
             ha='center', va='bottom', fontsize=12, fontweight='bold')

plt.tight_layout()
plt.show()

In [ ]:
# Show how much individual trees vary — and how the forest beats them all
individual_accs = np.array([
    accuracy_score(y_test, est.predict(X_test))
    for est in rf.estimators_
])

fig, ax = plt.subplots(figsize=(11, 4))
ax.hist(individual_accs * 100, bins=20, color='steelblue', edgecolor='white', alpha=0.8)
ax.axvline(individual_accs.mean()*100, color='orangered', lw=2,
           label=f'Average single-tree accuracy: {individual_accs.mean()*100:.1f}%')
ax.axvline(acc_rf*100, color='green', lw=2, ls='--',
           label=f'Forest accuracy: {acc_rf*100:.1f}%')
ax.set_xlabel('Individual tree accuracy (%)', fontsize=11)
ax.set_ylabel('Number of trees', fontsize=11)
ax.set_title('100 Individual Tree Accuracies vs Combined Forest Accuracy', fontsize=12)
ax.legend(fontsize=10)
plt.tight_layout()
plt.show()

print(f"Individual trees: min={individual_accs.min()*100:.1f}%  max={individual_accs.max()*100:.1f}%")
print(f"Forest combined : {acc_rf*100:.1f}% — typically beats the average individual tree")